# Clusterization of merged_features.csv

This notebook performs cluster analysis on the dataset using KMeans and HDBSCAN across four axes.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('merged_features.csv')
print(df.shape)
df.head()

## Axis 1: Financial Risk Profile
Features: credit_health, avg_credit_utilization, failed_txn_rate, atypical_txn_rate, dispute_rate, has_insurance, secured_card_flag
Expected clusters: Conservative, Moderate, Aggressive, Distressed

In [ ]:
# Select features
axis1_features = ['credit_health', 'avg_credit_utilization', 'failed_txn_rate', 'atypical_txn_rate', 'dispute_rate', 'has_insurance', 'secured_card_flag']
df_axis1 = df[axis1_features].dropna()

# Scale
scaler = RobustScaler()
X_axis1 = scaler.fit_transform(df_axis1)

# KMeans
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans_labels = kmeans.fit_predict(X_axis1)
print('KMeans Silhouette:', silhouette_score(X_axis1, kmeans_labels))

# HDBSCAN
hdb = hdbscan.HDBSCAN(min_cluster_size=5)
hdb_labels = hdb.fit_predict(X_axis1)
print('HDBSCAN Silhouette:', silhouette_score(X_axis1, hdb_labels) if len(set(hdb_labels)) > 1 else 'N/A')

# Visualize
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_axis1)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=kmeans_labels, palette='viridis')
plt.title('KMeans Clusters')
plt.subplot(1,2,2)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=hdb_labels, palette='viridis')
plt.title('HDBSCAN Clusters')
plt.show()

# Assign labels
cluster_means = df_axis1.groupby(kmeans_labels).mean()
# Risk score: higher values indicate higher risk (Distressed)
risk_score = (cluster_means['avg_credit_utilization'] + cluster_means['failed_txn_rate'] + 
              cluster_means['atypical_txn_rate'] + cluster_means['dispute_rate'] - 
              cluster_means['credit_health'] - cluster_means['has_insurance'] - 
              cluster_means['secured_card_flag'])
cluster_risk = risk_score.sort_values()
labels = ['Conservative', 'Moderate', 'Aggressive', 'Distressed']
cluster_to_label = {old: new for old, new in zip(cluster_risk.index, labels)}
df_axis1['kmeans_label'] = [cluster_to_label[c] for c in kmeans_labels]
print('KMeans labels:')
print(df_axis1['kmeans_label'].value_counts())

# For HDBSCAN
if len(set(hdb_labels)) == 4:
    hdb_cluster_means = df_axis1.groupby(hdb_labels).mean()
    hdb_risk_score = (hdb_cluster_means['avg_credit_utilization'] + hdb_cluster_means['failed_txn_rate'] + 
                      hdb_cluster_means['atypical_txn_rate'] + hdb_cluster_means['dispute_rate'] - 
                      hdb_cluster_means['credit_health'] - hdb_cluster_means['has_insurance'] - 
                      hdb_cluster_means['secured_card_flag'])
    hdb_cluster_risk = hdb_risk_score.sort_values()
    hdb_cluster_to_label = {old: new for old, new in zip(hdb_cluster_risk.index, labels)}
    df_axis1['hdb_label'] = [hdb_cluster_to_label[c] if c != -1 else 'Noise' for c in hdb_labels]
    print('HDBSCAN labels:')
    print(df_axis1['hdb_label'].value_counts())
else:
    print(f'HDBSCAN found {len(set(hdb_labels))} clusters (including noise if -1)')
    df_axis1['hdb_label'] = hdb_labels

## Axis 2: Income & Wealth Tier
Features: income_tier, max_credit_limit, investment_balance, monthly_avg_spend, cashback_total, has_investment, international_txn_rate
Expected clusters: Entry, Growing, Established, Affluent

In [ ]:
# Similar structure
axis2_features = ['income_tier', 'max_credit_limit', 'investment_balance', 'monthly_avg_spend', 'cashback_total', 'has_investment', 'international_txn_rate']
df_axis2 = df[axis2_features].dropna()
X_axis2 = scaler.fit_transform(df_axis2)

# KMeans
kmeans2 = KMeans(n_clusters=4, random_state=42)
kmeans_labels2 = kmeans2.fit_predict(X_axis2)
print('KMeans Silhouette:', silhouette_score(X_axis2, kmeans_labels2))

# HDBSCAN
hdb2 = hdbscan.HDBSCAN(min_cluster_size=5)
hdb_labels2 = hdb2.fit_predict(X_axis2)
print('HDBSCAN Silhouette:', silhouette_score(X_axis2, hdb_labels2) if len(set(hdb_labels2)) > 1 else 'N/A')

# Visualize
X_pca2 = pca.fit_transform(X_axis2)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=kmeans_labels2, palette='viridis')
plt.title('KMeans Clusters')
plt.subplot(1,2,2)
sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=hdb_labels2, palette='viridis')
plt.title('HDBSCAN Clusters')
plt.show()

# Assign labels
cluster_means2 = df_axis2.groupby(kmeans_labels2).mean()
# Wealth score: higher values indicate higher wealth
wealth_score = cluster_means2.sum(axis=1).sort_values(ascending=False)
labels2 = ['Affluent', 'Established', 'Growing', 'Entry']
cluster_to_label2 = {old: new for old, new in zip(wealth_score.index, labels2)}
df_axis2['kmeans_label'] = [cluster_to_label2[c] for c in kmeans_labels2]
print('KMeans labels:')
print(df_axis2['kmeans_label'].value_counts())

# For HDBSCAN
if len(set(hdb_labels2)) == 4:
    hdb_cluster_means2 = df_axis2.groupby(hdb_labels2).mean()
    hdb_wealth_score = hdb_cluster_means2.sum(axis=1).sort_values(ascending=False)
    hdb_cluster_to_label2 = {old: new for old, new in zip(hdb_wealth_score.index, labels2)}
    df_axis2['hdb_label'] = [hdb_cluster_to_label2[c] if c != -1 else 'Noise' for c in hdb_labels2]
    print('HDBSCAN labels:')
    print(df_axis2['hdb_label'].value_counts())
else:
    print(f'HDBSCAN found {len(set(hdb_labels2))} clusters (including noise if -1)')
    df_axis2['hdb_label'] = hdb_labels2

## Axis 3: Spending Lifestyle
Features: pct_supermercado, pct_restaurante, pct_entretenimiento, pct_viajes, pct_educacion, pct_salud, pct_tecnologia, pct_servicios_digitales, pct_ropa_accesorios, pct_transporte, pct_hogar, avg_ticket_size, msi_usage_rate, recurring_charge_count
Expected clusters: Essential spender, Foodie/Social, Tech/Digital native, Traveler, Family/Home, Status spender

In [ ]:
# Features
axis3_features = ['pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', 'pct_tecnologia', 'pct_servicios_digitales', 'pct_ropa_accesorios', 'pct_transporte', 'pct_hogar', 'avg_ticket_size', 'msi_usage_rate', 'recurring_charge_count']
df_axis3 = df[axis3_features].dropna()
X_axis3 = scaler.fit_transform(df_axis3)

# KMeans (6 clusters)
kmeans3 = KMeans(n_clusters=6, random_state=42)
kmeans_labels3 = kmeans3.fit_predict(X_axis3)
print('KMeans Silhouette:', silhouette_score(X_axis3, kmeans_labels3))

# HDBSCAN
hdb3 = hdbscan.HDBSCAN(min_cluster_size=5)
hdb_labels3 = hdb3.fit_predict(X_axis3)
print('HDBSCAN Silhouette:', silhouette_score(X_axis3, hdb_labels3) if len(set(hdb_labels3)) > 1 else 'N/A')

# Visualize
X_pca3 = pca.fit_transform(X_axis3)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.scatterplot(x=X_pca3[:,0], y=X_pca3[:,1], hue=kmeans_labels3, palette='viridis')
plt.title('KMeans Clusters')
plt.subplot(1,2,2)
sns.scatterplot(x=X_pca3[:,0], y=X_pca3[:,1], hue=hdb_labels3, palette='viridis')
plt.title('HDBSCAN Clusters')
plt.show()

# Assign labels
cluster_means3 = df_axis3.groupby(kmeans_labels3).mean()
# Spending score: sort by sum of features
spend_score = cluster_means3.sum(axis=1).sort_values()
labels3 = ['Essential spender', 'Foodie/Social', 'Tech/Digital native', 'Traveler', 'Family/Home', 'Status spender']
cluster_to_label3 = {old: new for old, new in zip(spend_score.index, labels3)}
df_axis3['kmeans_label'] = [cluster_to_label3[c] for c in kmeans_labels3]
print('KMeans labels:')
print(df_axis3['kmeans_label'].value_counts())

# For HDBSCAN
num_clusters_hdb3 = len(set(hdb_labels3)) - (1 if -1 in hdb_labels3 else 0)
if num_clusters_hdb3 == 6:
    hdb_cluster_means3 = df_axis3.groupby(hdb_labels3).mean()
    hdb_spend_score = hdb_cluster_means3.sum(axis=1).sort_values()
    hdb_cluster_to_label3 = {old: new for old, new in zip(hdb_spend_score.index, labels3)}
    df_axis3['hdb_label'] = [hdb_cluster_to_label3[c] if c != -1 else 'Noise' for c in hdb_labels3]
    print('HDBSCAN labels:')
    print(df_axis3['hdb_label'].value_counts())
else:
    print(f'HDBSCAN found {num_clusters_hdb3} clusters (excluding noise)')
    df_axis3['hdb_label'] = hdb_labels3

## Axis 4: Digital Engagement & Loyalty
Features: engagement_score, digital_payment_rate, cash_dependency, tenure_band, product_diversity_score, dias_desde_ultimo_login
Expected clusters: Power user, Casual, At-risk, Dormant

In [ ]:
# Features
axis4_features = ['engagement_score', 'digital_payment_rate', 'cash_dependency', 'tenure_band', 'product_diversity_score', 'dias_desde_ultimo_login']
df_axis4 = df[axis4_features].dropna()
X_axis4 = scaler.fit_transform(df_axis4)

# KMeans
kmeans4 = KMeans(n_clusters=4, random_state=42)
kmeans_labels4 = kmeans4.fit_predict(X_axis4)
print('KMeans Silhouette:', silhouette_score(X_axis4, kmeans_labels4))

# HDBSCAN
hdb4 = hdbscan.HDBSCAN(min_cluster_size=5)
hdb_labels4 = hdb4.fit_predict(X_axis4)
print('HDBSCAN Silhouette:', silhouette_score(X_axis4, hdb_labels4) if len(set(hdb_labels4)) > 1 else 'N/A')

# Visualize
X_pca4 = pca.fit_transform(X_axis4)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.scatterplot(x=X_pca4[:,0], y=X_pca4[:,1], hue=kmeans_labels4, palette='viridis')
plt.title('KMeans Clusters')
plt.subplot(1,2,2)
sns.scatterplot(x=X_pca4[:,0], y=X_pca4[:,1], hue=hdb_labels4, palette='viridis')
plt.title('HDBSCAN Clusters')
plt.show()

# Assign labels
cluster_means4 = df_axis4.groupby(kmeans_labels4).mean()
# Risk score: higher values indicate lower engagement (higher risk)
risk_score4 = (-cluster_means4['engagement_score'] - cluster_means4['digital_payment_rate'] + 
               cluster_means4['cash_dependency'] - cluster_means4['tenure_band'] - 
               cluster_means4['product_diversity_score'] + cluster_means4['dias_desde_ultimo_login'])
cluster_risk4 = risk_score4.sort_values()
labels4 = ['Power user', 'Casual', 'At-risk', 'Dormant']
cluster_to_label4 = {old: new for old, new in zip(cluster_risk4.index, labels4)}
df_axis4['kmeans_label'] = [cluster_to_label4[c] for c in kmeans_labels4]
print('KMeans labels:')
print(df_axis4['kmeans_label'].value_counts())

# For HDBSCAN
if len(set(hdb_labels4)) == 4:
    hdb_cluster_means4 = df_axis4.groupby(hdb_labels4).mean()
    hdb_risk_score4 = (-hdb_cluster_means4['engagement_score'] - hdb_cluster_means4['digital_payment_rate'] + 
                       hdb_cluster_means4['cash_dependency'] - hdb_cluster_means4['tenure_band'] - 
                       hdb_cluster_means4['product_diversity_score'] + hdb_cluster_means4['dias_desde_ultimo_login'])
    hdb_cluster_risk4 = hdb_risk_score4.sort_values()
    hdb_cluster_to_label4 = {old: new for old, new in zip(hdb_cluster_risk4.index, labels4)}
    df_axis4['hdb_label'] = [hdb_cluster_to_label4[c] if c != -1 else 'Noise' for c in hdb_labels4]
    print('HDBSCAN labels:')
    print(df_axis4['hdb_label'].value_counts())
else:
    print(f'HDBSCAN found {len(set(hdb_labels4))} clusters (including noise if -1)')
    df_axis4['hdb_label'] = hdb_labels4

## Summary

Summarize the results for each axis and algorithm.

In [ ]:
# Overall Summary
print('Axis 1: Financial Risk Profile')
print('KMeans Silhouette:', silhouette_score(X_axis1, kmeans_labels))
print('HDBSCAN Silhouette:', silhouette_score(X_axis1, hdb_labels) if len(set(hdb_labels)) > 1 else 'N/A')
print()
print('Axis 2: Income & Wealth Tier')
print('KMeans Silhouette:', silhouette_score(X_axis2, kmeans_labels2))
print('HDBSCAN Silhouette:', silhouette_score(X_axis2, hdb_labels2) if len(set(hdb_labels2)) > 1 else 'N/A')
print()
print('Axis 3: Spending Lifestyle')
print('KMeans Silhouette:', silhouette_score(X_axis3, kmeans_labels3))
print('HDBSCAN Silhouette:', silhouette_score(X_axis3, hdb_labels3) if len(set(hdb_labels3)) > 1 else 'N/A')
print()
print('Axis 4: Digital Engagement & Loyalty')
print('KMeans Silhouette:', silhouette_score(X_axis4, kmeans_labels4))
print('HDBSCAN Silhouette:', silhouette_score(X_axis4, hdb_labels4) if len(set(hdb_labels4)) > 1 else 'N/A')
